# Lab 2: Retrieval-Augmented Generation (RAG)

Objective: Build a minimal RAG pipeline using `transformers` with an encoder-only embedder (BERT) and a decoder-only generator (gemma 1B). We focus on understanding internal mechanisms, not efficiency.

This lab includes:
- A brief RAG overview
- Indexing Padua documents with BERT embeddings (token-based chunking)
- Retrieving top-n similar chunks by cosine similarity
- Generating short answers with gemma using retrieved context
- Exercises matching course goals

<style>
img[src="images/image.png"] {
    width: 30%,
    height: 20%;
}
</style>

![alt text](image.png)

_img from https://drjulija.github.io/posts/basic-rag/_

## Setup and Imports
We reuse patterns from Lab 1: local `cache_dir`, simple pooling for embeddings, and an optional gemma generation flag.

In [9]:
import os, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

print('Transformers version:', __import__('transformers').__version__)
print('Torch version:', torch.__version__)

BASE_DIR = os.path.join('../../lab2')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CACHE_DIR = os.path.join(BASE_DIR, 'models_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

RUN_gemma = False  # toggle to True to actually generate answers

# Model IDs (adjust if needed)
BERT_ID = 'bert-base-uncased'
GEMMA_ID = 'unsloth/gemma-3-1B-it'

Transformers version: 4.52.3
Torch version: 2.7.0+cu126


## Load Embedder (BERT) and Tokenizer
We use BERT as an encoder-only embedder. Embeddings are computed via mean pooling of the last hidden states across tokens.

In [10]:
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)
model_bert = AutoModel.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)
print('BERT special tokens:', tokenizer_bert.special_tokens_map)
print('BERT dtype:', next(model_bert.parameters()).dtype)
print('BERT device:', next(model_bert.parameters()).device)

BERT special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
BERT dtype: torch.float32
BERT device: cpu


## Load Generator (GEMMA)
We attempt to load a 1B chat model. If the primary model is unavailable, we fall back to Tinygemma 1.1B. Generation is optional and requires resources (GPU recommended).

In [11]:
def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=CACHE_DIR)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, cache_dir=CACHE_DIR, torch_dtype=torch.float16, device_map='auto'
    )
    return tok, mdl

tokenizer_gemma, model_gemma = load_model(GEMMA_ID)

print('gemma model:', GEMMA_ID)
print('Has chat template?', bool(getattr(tokenizer_gemma, 'chat_template', None)))

gemma model: unsloth/gemma-3-1B-it
Has chat template? True


## Load Documents, Embed them and save to Knowledge Base
We index 25 Padua documents.

In [12]:
docs_path = os.path.join(DATA_DIR, 'kb_docs.json')
with open(docs_path, 'r', encoding='utf-8') as f:
    DOCS = json.load(f)
print('there are', len(DOCS), 'docs')
DOCS[0]['title']

there are 25 docs


'Prato della Valle'

In [13]:
# Build KB chunks
KB_CHUNKS = []
for d in DOCS:
    text_to_encode = d['text']
    encodeing_input = tokenizer_bert(text_to_encode, return_tensors='pt', truncation=False, add_special_tokens=False)
    hidden_states = model_bert(**encodeing_input).last_hidden_state
    embedding = hidden_states.mean(dim=1).squeeze().detach().cpu().numpy()
    KB_CHUNKS.append({
        'doc_id': d['id'],
        'title': d['title'],
        'text': text_to_encode,
        'token_count': len(text_to_encode.split()),
        'embedding': embedding.tolist()
    })

index_path = os.path.join(DATA_DIR, 'kb_index.json')
with open(index_path, 'w', encoding='utf-8') as f:
    json.dump(KB_CHUNKS, f, ensure_ascii=False, indent=2)
len(KB_CHUNKS), KB_CHUNKS[0]['title']

(25, 'Prato della Valle')

## Retrieval: Top-n Similar Chunks
We compute cosine similarity between the query embedding and KB chunk embeddings and return the top-n chunks.

In [14]:
def embed_query(query):
    enc = tokenizer_bert([query], return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        out = model_bert(**enc)
        emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
    return emb.tolist()[0]

def cosine_similarity_matrix(query, kb_emb):
    query = np.array(query).reshape(1, -1)
    kb_emb = np.array(kb_emb)
    query = query / np.clip(np.linalg.norm(query, axis=1, keepdims=True), 1e-12, None)
    kb_emb = kb_emb / np.clip(np.linalg.norm(kb_emb, axis=1, keepdims=True), 1e-12, None)
    return query @ kb_emb.T

def retrieve_top_n(query, kb_emb, n=3):
    q = embed_query(query)
    document_embeddings = [chunk['embedding'] for chunk in kb_emb]
    sims = cosine_similarity_matrix(q, document_embeddings)[0]
    top_idx = np.argsort(-sims)[:n]
    return [{
        'text': kb_emb[i]['text'], 
        'similarity': float(sims[i])} 
        for i in top_idx]

In [15]:
# Load queries
queries_path = os.path.join(DATA_DIR, 'queries.json')
with open(queries_path, 'r', encoding='utf-8') as f:
    QUERIES = json.load(f)


# Demo retrieval on first query
q0 = QUERIES[0]['query']
print('Query:', q0)
top_chunks = retrieve_top_n(q0, KB_CHUNKS, n=3)
for retrieved in top_chunks:
    print(f'Similarity={retrieved["similarity"]:.3f}, text={retrieved["text"][:180]}')

Query: Where can you see Giotto’s frescoes in Padua?
Similarity=0.621, text=Founded in 1222, the University of Padua is renowned for research excellence, diverse faculties, and a vibrant international community.
Similarity=0.618, text=The Scrovegni Chapel contains Giotto’s famous frescoes. Entry is by ticket and timed slots, located near the Church of the Eremitani.
Similarity=0.598, text=Padua Cathedral (Duomo) stands alongside a Romanesque Baptistery with frescoes by Giusto de’ Menabuoi. Regular liturgical celebrations take place.


## Generation: Answer from Retrieved Context
We instruct GEMMA to answer concisely using only the provided context. If the answer is not in the context, the model should say it does not know.

In [ ]:
system_prompt = """You are a helpful assistant that answers questions about the city of Padova, Italy. 
Use the retrieved documents to answer the question as best as you can. 
If you don't know the answer, say you don't know."""


RESULTS = []
for query in QUERIES:
    top_chunks = retrieve_top_n(query['query'], KB_CHUNKS, n=3)
    for retrieved in top_chunks:
        print(f'Similarity={retrieved["similarity"]:.3f}, text={retrieved["text"][:180]}')
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query['query']},
    ]

    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)

    outputs = model_gemma.generate(**inputs, max_new_tokens=40)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:])
    RESULTS.append({
        'query': query['query'],
        'retrieved': top_chunks,
        'answer': answer
    })

## Excercise

1) You have generated an answer for each of the queries in `lab2/data/queries.json`. Each query has an `expected_answer`. Find a way to use Gemma to determine whether the generated answer is correct or not by prompting the LLM itself.

2) Create a RAG system that answer the queries in `lab2/data/excercise/queries.json` using the documents in `lab2/data/excercise/docs.txt`.


In [ ]:
cat > lab3/notebooks/lab2_rag_exercises_solution.ipynb <<'EOF'
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {
        "language": "markdown"
      },
      "source": [
        "# Lab 2 - Exercises Solution (Simple Version)",
        "",
        "This notebook contains simple reference solutions for the two final exercises:",
        "1. Use Gemma to judge whether generated answers are correct.",
        "2. Build a small RAG system on the football mini knowledge base."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "import os",
        "import json",
        "import numpy as np",
        "import torch",
        "from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM",
        "",
        "print('Transformers version:', __import__('transformers').__version__)",
        "print('Torch version:', torch.__version__)",
        "",
        "DATA_DIR = '../data'",
        "CACHE_DIR = './lab2/models_cache'",
        "os.makedirs(CACHE_DIR, exist_ok=True)",
        "",
        "BERT_ID = 'bert-base-uncased'",
        "GEMMA_ID = 'unsloth/gemma-3-1B-it'",
        "",
        "RUN_GEMMA = True  # Set to False if your machine cannot run generation"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "# Load embedding model (BERT)",
        "tokenizer_bert = AutoTokenizer.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)",
        "model_bert = AutoModel.from_pretrained(BERT_ID, cache_dir=CACHE_DIR)",
        "",
        "# Load generation model (Gemma)",
        "tokenizer_gemma = None",
        "model_gemma = None",
        "if RUN_GEMMA:",
        "    tokenizer_gemma = AutoTokenizer.from_pretrained(GEMMA_ID, cache_dir=CACHE_DIR)",
        "    model_gemma = AutoModelForCausalLM.from_pretrained(",
        "        GEMMA_ID,",
        "        cache_dir=CACHE_DIR,",
        "        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,",
        "        device_map='auto' if torch.cuda.is_available() else None",
        "    )",
        "",
        "print('BERT ready')",
        "print('Gemma ready:', RUN_GEMMA)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "def embed_text(text):",
        "    enc = tokenizer_bert([text], return_tensors='pt', padding=True, truncation=True)",
        "    with torch.no_grad():",
        "        out = model_bert(**enc)",
        "    return out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()",
        "",
        "def cosine_similarity(query_emb, doc_embs):",
        "    q = query_emb / np.clip(np.linalg.norm(query_emb), 1e-12, None)",
        "    d = doc_embs / np.clip(np.linalg.norm(doc_embs, axis=1, keepdims=True), 1e-12, None)",
        "    return d @ q",
        "",
        "def retrieve_top_n(query, kb, n=3):",
        "    q_emb = embed_text(query)",
        "    embs = np.array([x['embedding'] for x in kb])",
        "    sims = cosine_similarity(q_emb, embs)",
        "    top_ids = np.argsort(-sims)[:n]",
        "    return [{",
        "        'text': kb[i]['text'],",
        "        'similarity': float(sims[i])",
        "    } for i in top_ids]",
        "",
        "def generate_answer(question, retrieved_chunks, max_new_tokens=80):",
        "    if not RUN_GEMMA:",
        "        return 'Generation skipped because RUN_GEMMA=False'",
        "",
        "    context = '\\n\\n'.join([c['text'] for c in retrieved_chunks])",
        "    prompt = (",
        "        'Answer using only the context. If unknown, say: I do not know.\\n\\n'",
        "        f'Context:\\n{context}\\n\\nQuestion: {question}'",
        "    )",
        "    messages = [{"role": "user", "content": prompt}]",
        "",
        "    inputs = tokenizer_gemma.apply_chat_template(",
        "        messages,",
        "        add_generation_prompt=True,",
        "        tokenize=True,",
        "        return_tensors='pt',",
        "        return_dict=True",
        "    )",
        "    if torch.cuda.is_available():",
        "        inputs = {k: v.to(model_gemma.device) for k, v in inputs.items()}",
        "",
        "    with torch.no_grad():",
        "        outputs = model_gemma.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)",
        "",
        "    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]",
        "    return tokenizer_gemma.decode(new_tokens, skip_special_tokens=True).strip()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "language": "markdown"
      },
      "source": [
        "## Exercise 1: Let Gemma Judge Correctness",
        "",
        "Pipeline:",
        "1. Retrieve context for each Padua query.",
        "2. Generate an answer.",
        "3. Ask Gemma if generated answer matches expected answer."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "# Build Padua KB with one embedding per document",
        "with open(os.path.join(DATA_DIR, 'kb_docs.json'), 'r', encoding='utf-8') as f:",
        "    padua_docs = json.load(f)",
        "",
        "KB_PADUA = []",
        "for d in padua_docs:",
        "    emb = embed_text(d['text'])",
        "    KB_PADUA.append({",
        "        'id': d['id'],",
        "        'text': d['text'],",
        "        'embedding': emb",
        "    })",
        "",
        "with open(os.path.join(DATA_DIR, 'queries.json'), 'r', encoding='utf-8') as f:",
        "    padua_queries = json.load(f)",
        "",
        "def judge_answer_with_gemma(question, expected, predicted):",
        "    if not RUN_GEMMA:",
        "        return {'verdict': 'UNKNOWN', 'reason': 'Gemma not running'}",
        "",
        "    prompt = f'''You are a strict grader.",
        "Question: {question}",
        "Expected answer: {expected}",
        "Student answer: {predicted}",
        "",
        "Reply with exactly one line in this format:",
        "VERDICT: CORRECT",
        "or",
        "VERDICT: INCORRECT",
        "Then one short reason on a second line starting with REASON:.'''",
        "",
        "    messages = [{"role": "user", "content": prompt}]",
        "    inputs = tokenizer_gemma.apply_chat_template(",
        "        messages,",
        "        add_generation_prompt=True,",
        "        tokenize=True,",
        "        return_tensors='pt',",
        "        return_dict=True",
        "    )",
        "    if torch.cuda.is_available():",
        "        inputs = {k: v.to(model_gemma.device) for k, v in inputs.items()}",
        "",
        "    with torch.no_grad():",
        "        outputs = model_gemma.generate(**inputs, max_new_tokens=50, do_sample=False)",
        "",
        "    raw = tokenizer_gemma.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()",
        "    first = raw.splitlines()[0].upper() if raw else ''",
        "    verdict = 'CORRECT' if 'CORRECT' in first and 'INCORRECT' not in first else 'INCORRECT'",
        "    reason = raw.splitlines()[1] if len(raw.splitlines()) > 1 else ''",
        "    return {'verdict': verdict, 'reason': reason, 'raw': raw}",
        "",
        "results_ex1 = []",
        "for item in padua_queries:",
        "    q = item['query']",
        "    expected = item['expected_answer']",
        "    retrieved = retrieve_top_n(q, KB_PADUA, n=3)",
        "    predicted = generate_answer(q, retrieved, max_new_tokens=60)",
        "    judgment = judge_answer_with_gemma(q, expected, predicted)",
        "",
        "    results_ex1.append({",
        "        'query': q,",
        "        'expected': expected,",
        "        'predicted': predicted,",
        "        'verdict': judgment['verdict'],",
        "        'reason': judgment.get('reason', '')",
        "    })",
        "",
        "for r in results_ex1:",
        "    print('Q:', r['query'])",
        "    print('Expected:', r['expected'])",
        "    print('Predicted:', r['predicted'])",
        "    print('Verdict:', r['verdict'])",
        "    print('Reason:', r['reason'])",
        "    print('-' * 80)",
        "",
        "if len(results_ex1) > 0:",
        "    acc = sum(1 for r in results_ex1 if r['verdict'] == 'CORRECT') / len(results_ex1)",
        "    print('Gemma-judge accuracy estimate:', round(acc, 3))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "language": "markdown"
      },
      "source": [
        "## Exercise 2: RAG on Football Rules",
        "",
        "We read the football text file, split it into simple chunks, retrieve top chunks, and answer each query."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "# Load football docs and queries",
        "with open(os.path.join(DATA_DIR, 'excercise', 'docs.txt'), 'r', encoding='utf-8') as f:",
        "    football_text = f.read()",
        "",
        "with open(os.path.join(DATA_DIR, 'excercise', 'queries.json'), 'r', encoding='utf-8') as f:",
        "    football_queries = json.load(f)",
        "",
        "# Very simple chunking: split by empty lines",
        "raw_chunks = [c.strip() for c in football_text.split('\\n\\n') if c.strip()]",
        "",
        "KB_FOOTBALL = []",
        "for i, chunk in enumerate(raw_chunks):",
        "    KB_FOOTBALL.append({",
        "        'id': i,",
        "        'text': chunk,",
        "        'embedding': embed_text(chunk)",
        "    })",
        "",
        "print('Football chunks:', len(KB_FOOTBALL))"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "execution_count": null,
      "outputs": [],
      "source": [
        "results_ex2 = []",
        "",
        "for item in football_queries:",
        "    q = item['query']",
        "    expected = item['expected_answer']",
        "",
        "    retrieved = retrieve_top_n(q, KB_FOOTBALL, n=3)",
        "    predicted = generate_answer(q, retrieved, max_new_tokens=70)",
        "",
        "    # Basic automatic check: expected phrase appears in answer",
        "    hit = expected.lower() in predicted.lower()",
        "",
        "    results_ex2.append({",
        "        'query': q,",
        "        'expected': expected,",
        "        'predicted': predicted,",
        "        'matched_expected_substring': hit",
        "    })",
        "",
        "for r in results_ex2:",
        "    print('Q:', r['query'])",
        "    print('Expected:', r['expected'])",
        "    print('Predicted:', r['predicted'])",
        "    print('Expected in prediction?', r['matched_expected_substring'])",
        "    print('-' * 80)",
        "",
        "if len(results_ex2) > 0:",
        "    hit_rate = sum(1 for r in results_ex2 if r['matched_expected_substring']) / len(results_ex2)",
        "    print('Substring match rate:', round(hit_rate, 3))"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
EOF
